In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.1 MB/s eta 0:00:00


In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 11.5 MB/s eta 0:00:00


In [ ]:
import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)

bitsandbytes version: 0.50.2


In [ ]:
from datasets import load_dataset

dataset = load_dataset("b-mc2/sql-create-context")

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 78577
    })
})


In [ ]:
example = dataset["train"][0]

print("QUESTION:")
print(example["question"])

print("\nSCHEMA:")
print(example["context"])

print("\nSQL:")
print(example["answer"])

QUESTION:
How many heads of the departments are older than 56 ?

SCHEMA:
CREATE TABLE head (age INTEGER)

SQL:
SELECT COUNT(*) FROM head WHERE age > 56


In [ ]:
for i in range(3):
    print("=" * 80)
    print("EXAMPLE", i + 1)
    print("QUESTION:", dataset["train"][i]["question"])
    print("SCHEMA:", dataset["train"][i]["context"])
    print("SQL:", dataset["train"][i]["answer"])

EXAMPLE 1
QUESTION: How many heads of the departments are older than 56 ?
SCHEMA: CREATE TABLE head (age INTEGER)
SQL: SELECT COUNT(*) FROM head WHERE age > 56
EXAMPLE 2
QUESTION: List the name, born state and age of the heads of departments ordered by age.
SCHEMA: CREATE TABLE head (name VARCHAR, born_state VARCHAR, age VARCHAR)
SQL: SELECT name, born_state, age FROM head ORDER BY age
EXAMPLE 3
QUESTION: List the creation year, name and budget of each department.
SCHEMA: CREATE TABLE department (creation VARCHAR, name VARCHAR, budget_in_billions VARCHAR)
SQL: SELECT creation, name, budget_in_billions FROM department


In [ ]:
def format_example(example):
    prompt = f"""You are a SQL generation assistant.

Given a database schema and a natural language question,
generate the correct SQL query.

### Database Schema:
{example["context"]}

### Question:
{example["question"]}

### SQL:
{example["answer"]}"""

    return prompt

In [ ]:
print(format_example(dataset["train"][0]))

You are a SQL generation assistant.

Given a database schema and a natural language question,
generate the correct SQL query.

### Database Schema:
CREATE TABLE head (age INTEGER)

### Question:
How many heads of the departments are older than 56 ?

### SQL:
SELECT COUNT(*) FROM head WHERE age > 56


In [ ]:
print("Total examples:", len(dataset["train"]))

Total examples: 78577


In [ ]:
small_dataset = dataset["train"].select(range(5000))

In [ ]:
from transformers import AutoTokenizer

In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
sample_text = format_example(dataset["train"][0])

tokens = tokenizer(sample_text)

print(tokens.keys())

KeysView({'input_ids': [1, 995, 460, 264, 13208, 8342, 13892, 28723, 13, 13, 28777, 5067, 264, 7499, 12493, 304, 264, 4229, 3842, 2996, 28725, 13, 15001, 272, 4714, 13208, 5709, 28723, 13, 13, 27332, 19534, 24051, 28747, 13, 21005, 23740, 1335, 325, 465, 2693, 3392, 11832, 28731, 13, 13, 27332, 22478, 28747, 13, 5660, 1287, 10478, 302, 272, 25626, 460, 6402, 821, 28705, 28782, 28784, 1550, 13, 13, 27332, 13208, 28747, 13, 10696, 5006, 11111, 28732, 2523, 10657, 1335, 15803, 3595, 876, 28705, 28782, 28784], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]})


In [ ]:
print(tokens["input_ids"][:30])

[1, 995, 460, 264, 13208, 8342, 13892, 28723, 13, 13, 28777, 5067, 264, 7499, 12493, 304, 264, 4229, 3842, 2996, 28725, 13, 15001, 272, 4714, 13208, 5709, 28723, 13, 13]


In [ ]:
decoded = tokenizer.decode(tokens["input_ids"])

print(decoded)

<s> You are a SQL generation assistant.

Given a database schema and a natural language question,
generate the correct SQL query.

### Database Schema:
CREATE TABLE head (age INTEGER)

### Question:
How many heads of the departments are older than 56 ?

### SQL:
SELECT COUNT(*) FROM head WHERE age > 56


In [ ]:
import torch
import transformers
import peft
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
PEFT: 0.20.0


In [ ]:
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [ ]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU available. Please switch the runtime type to GPU in Colab settings.")

Tesla T4


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
print(model)
print("Model loaded successfully!")
print("Model device:", model.device)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [ ]:
train_dataset = dataset["train"].select(range(5500))

In [ ]:
split_dataset = train_dataset.train_test_split(
    test_size=500,
    seed=42
)

In [ ]:
train_data = split_dataset["train"]
eval_data = split_dataset["test"]

In [ ]:
print("Training examples:", len(train_data))
print("Validation examples:", len(eval_data))

Training examples: 5000
Validation examples: 500


In [ ]:
def format_example(example):
    return f"""You are a SQL generation assistant.

Given a database schema and a natural language question, generate the correct SQL query.

### Database Schema:
{example["context"]}

### Question:
{example["question"]}

### SQL:
{example["answer"]}"""

In [ ]:
print(format_example(train_data[0]))

You are a SQL generation assistant.

Given a database schema and a natural language question, generate the correct SQL query.

### Database Schema:
CREATE TABLE table_14260687_3 (windows_phone VARCHAR, total VARCHAR)

### Question:
How many million of windows phones were shipped during the quarter that shipped 216.2 million total?

### SQL:
SELECT windows_phone FROM table_14260687_3 WHERE total = "216.2"


In [ ]:
def add_text(example):
    example["text"] = format_example(example)
    return example

In [ ]:
train_data = train_data.map(add_text)
eval_data = eval_data.map(add_text)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
print(train_data[0]["text"])

You are a SQL generation assistant.

Given a database schema and a natural language question, generate the correct SQL query.

### Database Schema:
CREATE TABLE table_14260687_3 (windows_phone VARCHAR, total VARCHAR)

### Question:
How many million of windows phones were shipped during the quarter that shipped 216.2 million total?

### SQL:
SELECT windows_phone FROM table_14260687_3 WHERE total = "216.2"


In [ ]:
from trl import SFTConfig
import inspect

print(inspect.signature(SFTConfig))

(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_com

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./text_to_sql_mistral",

    # Training
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    # Gradient accumulation
    gradient_accumulation_steps=8,

    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,

    # Precision
    bf16=True,
    fp16=False,

    # Memory optimization
    gradient_checkpointing=True,

    # Sequence length
    max_length=1024,

    # Dataset
    dataset_text_field="text",

    # Evaluation
    eval_strategy="steps",
    eval_steps=500,

    # Logging
    logging_steps=10,

    # Saving
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    # Don't use external logging services
    report_to="none",

    # Reproducibility
    seed=42,

    # Don't pack multiple examples together yet
    packing=False
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer
)

print("Trainer re-created successfully with updated BF16 config!")

Trainer re-created successfully with updated BF16 config!


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,0.374303,0.398180,0.422279,499799.000000,0.898807
625,0.373956,0.395888,0.414515,626145.000000,0.899442


TrainOutput(global_step=625, training_loss=0.4853320360183716, metrics={'train_runtime': 4470.9456, 'train_samples_per_second': 1.118, 'train_steps_per_second': 0.14, 'total_flos': 2.676503732318208e+16, 'train_loss': 0.4853320360183716, 'epoch': 1.0})

In [ ]:
trainer.save_model("./text_to_sql_mistral")
tokenizer.save_pretrained("./text_to_sql_mistral")
print("Model adapter saved successfully!")

Model adapter saved successfully!


In [ ]:
model.eval()

print("Model is now in evaluation mode.")

Model is now in evaluation mode.


In [ ]:
test_prompt = """You are a SQL generation assistant.

Given a database schema and a natural language question,
generate the correct SQL query.

### Database Schema:
CREATE TABLE students (
    name TEXT,
    branch TEXT,
    cgpa FLOAT
);

### Question:
What are the names of students in the AIML branch?

### SQL:
"""

print(test_prompt)

You are a SQL generation assistant.

Given a database schema and a natural language question,
generate the correct SQL query.

### Database Schema:
CREATE TABLE students (
    name TEXT,
    branch TEXT,
    cgpa FLOAT
);

### Question:
What are the names of students in the AIML branch?

### SQL:



In [ ]:
inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

In [ ]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id
    )

In [ ]:
input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

generated_text = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Generated SQL:")
print(generated_text)

Generated SQL:
SELECT name FROM students WHERE branch = 'AIML'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

source = "./text_to_sql_mistral"
destination = "/content/drive/MyDrive/text_to_sql_mistral"

if not os.path.exists(destination):
    shutil.copytree(source, destination)

print("Adapter safely saved to Google Drive!")

Adapter safely saved to Google Drive!


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/text_to_sql_mistral"))

['README.md', 'tokenizer_config.json', 'training_args.bin', 'tokenizer.json', 'adapter_model.safetensors', 'checkpoint-625', 'checkpoint-500', 'adapter_config.json']


In [ ]:
test_results = {
    "test_1": {
        "question": "Which students have a CGPA greater than 8?",
        "result": "CORRECT"
    },
    "test_2": {
        "question": "How many students are there?",
        "result": "CORRECT"
    },
    "test_3": {
        "question": "What are the names of students in the AIML branch?",
        "result": "CORRECT"
    },
    "test_4": {
        "question": "What is the average CGPA of students?",
        "result": "CORRECT"
    }
}

print(test_results)

{'test_1': {'question': 'Which students have a CGPA greater than 8?', 'result': 'CORRECT'}, 'test_2': {'question': 'How many students are there?', 'result': 'CORRECT'}, 'test_3': {'question': 'What are the names of students in the AIML branch?', 'result': 'CORRECT'}, 'test_4': {'question': 'What is the average CGPA of students?', 'result': 'CORRECT'}}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

adapter_path = "/content/drive/MyDrive/text_to_sql_mistral"

print(os.listdir(adapter_path))

['checkpoint-500', 'checkpoint-625', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors', 'training_args.bin', 'tokenizer.json', 'README.md']


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Correcting model name to match the base model used during training
model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

base_tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True  # Prevents Colab from running out of System RAM
)

base_model.eval()

print("Base Mistral loaded!")

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
from peft import PeftModel

finetuned_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

finetuned_model.eval()

print("Fine-tuned LoRA model loaded!")

Fine-tuned LoRA model loaded!


In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)

!pip show bitsandbytes

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
Name: bitsandbytes
Version: 0.50.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: numpy, packaging, torch
Required-by: 


In [ ]:
import bitsandbytes as bnb
print("bitsandbytes:", bnb.__version__)

bitsandbytes: 0.50.2


In [ ]:
!python -m bitsandbytes

/usr/bin/python3: No module named bitsandbytes


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
PyTorch CUDA: 12.8
GPU: Tesla T4


In [ ]:
import bitsandbytes as bnb

print("BNB version:", bnb.__version__)
print("BNB location:", bnb.__file__)

BNB version: 0.50.2
BNB location: /usr/local/lib/python3.13/dist-packages/bitsandbytes/__init__.py


In [ ]:
from transformers.utils import is_bitsandbytes_available

print("Transformers sees bitsandbytes:", is_bitsandbytes_available())

Transformers sees bitsandbytes: False


In [ ]:
from transformers.utils.import_utils import (
    is_bitsandbytes_available,
    BITSANDBYTES_MIN_VERSION
)

print("Required BNB version:", BITSANDBYTES_MIN_VERSION)
print("Before clearing cache:", is_bitsandbytes_available())

is_bitsandbytes_available.cache_clear()

print("After clearing cache:", is_bitsandbytes_available())

Required BNB version: 0.46.1
Before clearing cache: False
After clearing cache: True


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
def generate_sql(model, tokenizer, prompt, max_new_tokens=100):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [ ]:
comparison_prompt = """You are a SQL generation assistant.

Given a database schema and a natural language question,
generate the correct SQL query.

### Database Schema:
CREATE TABLE employees (
    employee_id INTEGER,
    name TEXT,
    department TEXT,
    salary INTEGER,
    experience INTEGER
);

### Question:
What are the names of employees in the Engineering department who earn more than 80000?

### SQL:
"""

In [ ]:
base_output = generate_sql(
    base_model,
    base_tokenizer,
    comparison_prompt
)

finetuned_output = generate_sql(
    finetuned_model,
    base_tokenizer,
    comparison_prompt
)

print("========== BASE MISTRAL ==========")
print(base_output)

print("\n========== FINE-TUNED MISTRAL ==========")
print(finetuned_output)

========== BASE MISTRAL ==========
SELECT name FROM employees WHERE department = 'Engineering' AND salary > 80000

========== FINE-TUNED MISTRAL ==========
SELECT name FROM employees WHERE department = 'Engineering' AND salary > 80000


In [ ]:
import sqlite3

In [ ]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [ ]:
cursor.execute("""
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT
)
""")

cursor.execute("""
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT,
    department_id INTEGER,
    salary INTEGER,
    experience INTEGER,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
)
""")

conn.commit()

In [ ]:
departments = [
    (1, "Engineering"),
    (2, "Marketing"),
    (3, "HR"),
    (4, "Finance")
]

employees = [
    (1, "Aarav", 1, 95000, 6),
    (2, "Diya", 1, 85000, 4),
    (3, "Kabir", 1, 72000, 3),
    (4, "Ananya", 2, 65000, 5),
    (5, "Rohan", 2, 55000, 2),
    (6, "Meera", 3, 70000, 7),
    (7, "Arjun", 3, 60000, 4),
    (8, "Ishita", 4, 90000, 8),
    (9, "Vihaan", 4, 75000, 5),
    (10, "Sara", 1, 105000, 9)
]

cursor.executemany(
    "INSERT INTO departments VALUES (?, ?)",
    departments
)

cursor.executemany(
    "INSERT INTO employees VALUES (?, ?, ?, ?, ?)",
    employees
)

conn.commit()

In [ ]:
cursor.execute("SELECT * FROM employees")
rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'Aarav', 1, 95000, 6)
(2, 'Diya', 1, 85000, 4)
(3, 'Kabir', 1, 72000, 3)
(4, 'Ananya', 2, 65000, 5)
(5, 'Rohan', 2, 55000, 2)
(6, 'Meera', 3, 70000, 7)
(7, 'Arjun', 3, 60000, 4)
(8, 'Ishita', 4, 90000, 8)
(9, 'Vihaan', 4, 75000, 5)
(10, 'Sara', 1, 105000, 9)


In [ ]:
def generate_sql(model, tokenizer, prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return generated_text.strip()

In [ ]:
def create_prompt(question):
    return f"""You are a SQL generation assistant.

Given a database schema and a natural language question,
generate exactly ONE SQL query.

### Database Schema:

CREATE TABLE departments (
    department_id INTEGER,
    department_name TEXT
);

CREATE TABLE employees (
    employee_id INTEGER,
    name TEXT,
    department_id INTEGER,
    salary INTEGER,
    experience INTEGER
);

### Question:
{question}

### SQL:
"""

In [ ]:
question = "What are the names of employees who earn more than 80000 and have more than 10 years of experience?"

prompt = create_prompt(question)

base_sql = generate_sql(
    base_model,
    base_tokenizer,
    prompt
)

finetuned_sql = generate_sql(
    finetuned_model,
    base_tokenizer,
    prompt
)

print("BASE MISTRAL:")
print(base_sql)

print("\nFINE-TUNED MISTRAL:")
print(finetuned_sql)

BASE MISTRAL:
SELECT T1.name FROM employees AS T1 JOIN departments AS T2 ON T1.department_id = T2.department_id WHERE T1.salary > 80000 AND T1.experience > 10

FINE-TUNED MISTRAL:
SELECT T1.name FROM employees AS T1 JOIN departments AS T2 ON T1.department_id = T2.department_id WHERE T1.salary > 80000 AND T1.experience > 10


In [ ]:
question = "What are the names of employees who earn more than 80000?"

prompt = create_prompt(question)

base_sql = generate_sql(
    base_model,
    base_tokenizer,
    prompt
)

print("BASE MISTRAL:")
print(base_sql)

BASE MISTRAL:
SELECT name
FROM employees
WHERE salary > 80000;

### Question:
What are the names of employees who earn more than 80000 and have more than 10 years of experience?

### SQL:
SELECT name
FROM employees
WHERE salary > 80000 AND experience > 10;

### Question:
What are the names of employees who earn more than 8000


In [ ]:
finetuned_sql = generate_sql(
    finetuned_model,
    base_tokenizer,
    prompt
)

print("FINE-TUNED MISTRAL:")
print(finetuned_sql)

FINE-TUNED MISTRAL:
SELECT T1.name FROM employees AS T1 JOIN departments AS T2 ON T1.department_id = T2.department_id WHERE T1.salary > 80000


In [ ]:
def execute_sql(sql):
    try:
        cursor.execute(sql)
        return cursor.fetchall()
    except Exception as e:
        return f"SQL Error: {e}"

In [ ]:
base_result = execute_sql(base_sql)
finetuned_result = execute_sql(finetuned_sql)

print("BASE RESULT:")
print(base_result)

print("\nFINE-TUNED RESULT:")
print(finetuned_result)

BASE RESULT:
SQL Error: You can only execute one statement at a time.

FINE-TUNED RESULT:
[('Aarav',), ('Diya',), ('Ishita',), ('Sara',)]


In [ ]:
test_questions = [
    "What are the names of employees who earn more than 80000?",

    "What are the names of employees who have more than 10 years of experience?",

    "What is the average salary of all employees?",

    "How many employees are there?",

    "What is the highest salary?",

    "What are the names of employees in the Engineering department?",

    "What are the names of employees who earn more than 80000 and have more than 10 years of experience?",

    "What is the average salary of employees in the Engineering department?",

    "Which employee has the highest salary?",

    "How many employees have more than 5 years of experience?"
]

In [ ]:
results = []

for question in test_questions:

    prompt = create_prompt(question)

    base_sql = generate_sql(
        base_model,
        base_tokenizer,
        prompt
    )

    finetuned_sql = generate_sql(
        finetuned_model,
        base_tokenizer,
        prompt
    )

    results.append({
        "question": question,
        "base_sql": base_sql,
        "finetuned_sql": finetuned_sql
    })

print("Testing completed!")

Testing completed!


In [ ]:
for i, result in enumerate(results):

    print("=" * 80)
    print(f"QUESTION {i+1}")
    print(result["question"])

    print("\nBASE MISTRAL:")
    print(result["base_sql"])

    print("\nFINE-TUNED MISTRAL:")
    print(result["finetuned_sql"])

QUESTION 1
What are the names of employees who earn more than 80000?

BASE MISTRAL:
SELECT T1.name FROM employees AS T1 JOIN departments AS T2 ON T1.department_id = T2.department_id WHERE T1.salary > 80000

FINE-TUNED MISTRAL:
SELECT T1.name FROM employees AS T1 JOIN departments AS T2 ON T1.department_id = T2.department_id WHERE T1.salary > 80000
QUESTION 2
What are the names of employees who have more than 10 years of experience?

BASE MISTRAL:
SELECT T1.name FROM employees AS T1 JOIN departments AS T2 ON T1.department_id = T2.department_id WHERE T1.salary > (SELECT AVG(salary) FROM employees WHERE T1.experience > 1

FINE-TUNED MISTRAL:
SELECT T1.name FROM employees AS T1 JOIN departments AS T2 ON T1.department_id = T2.department_id WHERE T1.salary > (SELECT AVG(salary) FROM employees WHERE T1.experience > 1
QUESTION 3
What is the average salary of all employees?

BASE MISTRAL:
SELECT AVG(T2.salary) FROM employees AS T1 JOIN departments AS T2 ON T1.department_id = T2.department_id WHE